We start off by loading in all the data

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.special import expit

DATA_DIR = Path(".")

X_trn_df = pd.read_csv(DATA_DIR / "X_trn.csv")
y_trn_df = pd.read_csv(DATA_DIR / "y_trn.csv")
X_test_df = pd.read_csv(DATA_DIR / "X_test.csv")

X_trn = X_trn_df.to_numpy(dtype=np.float64)
Y_trn = y_trn_df.to_numpy(dtype=np.float64)
X_test = X_test_df.to_numpy(dtype=np.float64)

print(X_trn_df.shape)
print(y_trn_df.shape)
print(X_test_df.shape)

(25000, 23)
(25000, 1)
(5000, 23)


Here we have the full model for the Brier Score

In [ ]:
import numpy as np
from scipy.special import expit

# -----------------------------
# Helpers
# -----------------------------

def train_val_split(X, Y, val_fraction=0.2, seed=42):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    indices = rng.permutation(n)

    n_val = int(val_fraction * n)
    val_idx = indices[:n_val]
    train_idx = indices[n_val:]

    return X[train_idx], Y[train_idx], X[val_idx], Y[val_idx]


def fit_standardizer(X):
    mu = X.mean(axis=0, keepdims=True)
    sigma = X.std(axis=0, keepdims=True)
    sigma[sigma == 0] = 1.0
    return mu, sigma


def apply_standardizer(X, mu, sigma):
    return (X - mu) / sigma


def initialize_parameters(n_features, n_hidden, seed=42):
    rng = np.random.default_rng(seed)

    W1 = rng.normal(0.0, np.sqrt(2 / n_features), size=(n_features, n_hidden))
    b1 = np.zeros((1, n_hidden))

    W2 = rng.normal(0.0, np.sqrt(2 / n_hidden), size=(n_hidden, 1))
    b2 = np.zeros((1, 1))

    return W1, b1, W2, b2


def predict_neural_network(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1
    h = np.maximum(0, z1)

    z2 = h @ W2 + b2
    p = expit(z2)

    return p


def brier_score(Y, p):
    return np.mean((Y - p) ** 2)


def binary_cross_entropy_from_logits(Y, z):
    # Stable BCE: no log(0), even when the sigmoid rounds to 0 or 1.
    return np.mean(np.logaddexp(0, z) - Y * z)


def train_neural_network_minibatch_bce(
    X,
    Y,
    W1,
    b1,
    W2,
    b2,
    eta=0.01,
    decay=0.0,
    iterations=1000,
    batch_size=256,
    seed=42
):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    Loss = []

    for epoch in range(iterations):
        eta_epoch = eta / (1 + decay * epoch)

        indices = rng.permutation(n)

        for start in range(0, n, batch_size):
            batch_idx = indices[start:start + batch_size]

            X_batch = X[batch_idx]
            Y_batch = Y[batch_idx]
            m = X_batch.shape[0]

            # Forward pass
            z1 = X_batch @ W1 + b1
            h = np.maximum(0, z1)

            z2 = h @ W2 + b2
            p = expit(z2)

            # Unweighted BCE backward pass (sigmoid output).
            dz2 = (p - Y_batch) / m

            gradient_W2 = h.T @ dz2
            gradient_b2 = np.sum(dz2, axis=0, keepdims=True)

            relu_derivative = (z1 > 0).astype(float)
            dz1 = (dz2 @ W2.T) * relu_derivative

            gradient_W1 = X_batch.T @ dz1
            gradient_b1 = np.sum(dz1, axis=0, keepdims=True)

            # Update
            W1 -= eta_epoch * gradient_W1
            b1 -= eta_epoch * gradient_b1
            W2 -= eta_epoch * gradient_W2
            b2 -= eta_epoch * gradient_b2

        # Loss tracks training BCE; Brier is used separately for validation.
        h_train = np.maximum(0, X @ W1 + b1)
        z_train = h_train @ W2 + b2
        Loss.append(binary_cross_entropy_from_logits(Y, z_train))

    return W1, b1, W2, b2, Loss


# -----------------------------
# 1. Train/validation split
# -----------------------------

# Keep labels as a column vector to match the predictions.
Y_trn = np.asarray(Y_trn, dtype=float).reshape(-1, 1)

X_fit_raw, Y_fit, X_val_raw, Y_val = train_val_split(
    X_trn,
    Y_trn,
    val_fraction=0.2,
    seed=42
)

# -----------------------------
# 2. Scaling fitted only on fit/train split
# -----------------------------

mu_val, sigma_val = fit_standardizer(X_fit_raw)

X_fit = apply_standardizer(X_fit_raw, mu_val, sigma_val)
X_val = apply_standardizer(X_val_raw, mu_val, sigma_val)

# -----------------------------
# 3. Hyperparameter search
# -----------------------------

eta_grid = [1.0, 0.3, 0.1, 0.03, 0.01]
decay_grid = [0.0, 0.001, 0.003, 0.01]
n_hidden_grid = [2, 4, 8, 16]
batch_size_grid = [128, 256, 512]

iterations = 1000
n_features = X_fit.shape[1]

results_brier = []

for n_hidden in n_hidden_grid:
    for batch_size in batch_size_grid:
        for eta in eta_grid:
            for decay in decay_grid:
                W1, b1, W2, b2 = initialize_parameters(
                    n_features=n_features,
                    n_hidden=n_hidden,
                    seed=42
                )

                W1, b1, W2, b2, Loss = train_neural_network_minibatch_bce(
                    X_fit,
                    Y_fit,
                    W1,
                    b1,
                    W2,
                    b2,
                    eta=eta,
                    decay=decay,
                    iterations=iterations,
                    batch_size=batch_size,
                    seed=42
                )

                p_val = predict_neural_network(X_val, W1, b1, W2, b2)
                val_brier = brier_score(Y_val, p_val)

                train_bce = Loss[-1]

                results_brier.append({
                    "n_hidden": n_hidden,
                    "batch_size": batch_size,
                    "eta": eta,
                    "decay": decay,
                    "val_brier": val_brier,
                    "train_bce": train_bce
                })

                print(
                    "n_hidden:", n_hidden,
                    "| batch_size:", batch_size,
                    "| eta:", eta,
                    "| decay:", decay,
                    "| validation Brier:", val_brier,
                    "| train BCE:", train_bce
                )

# -----------------------------
# 4. Select best hyperparameters
# -----------------------------

best_result_brier = min(results_brier, key=lambda x: x["val_brier"])

best_n_hidden_brier = best_result_brier["n_hidden"]
best_batch_size_brier = best_result_brier["batch_size"]
best_eta_brier = best_result_brier["eta"]
best_decay_brier = best_result_brier["decay"]

print("\nBest Brier result:")
print(best_result_brier)

# -----------------------------
# 5. Final BCE model on all training data, selected on validation Brier
# -----------------------------

mu_final_brier, sigma_final_brier = fit_standardizer(X_trn)

X_trn_scaled_brier = apply_standardizer(X_trn, mu_final_brier, sigma_final_brier)
X_test_scaled_brier = apply_standardizer(X_test, mu_final_brier, sigma_final_brier)

W1_brier, b1_brier, W2_brier, b2_brier = initialize_parameters(
    n_features=X_trn_scaled_brier.shape[1],
    n_hidden=best_n_hidden_brier,
    seed=42
)

W1_brier, b1_brier, W2_brier, b2_brier, Loss_final_brier = train_neural_network_minibatch_bce(
    X_trn_scaled_brier,
    Y_trn,
    W1_brier,
    b1_brier,
    W2_brier,
    b2_brier,
    eta=best_eta_brier,
    decay=best_decay_brier,
    iterations=iterations,
    batch_size=best_batch_size_brier,
    seed=42
)

p_test_brier = predict_neural_network(
    X_test_scaled_brier,
    W1_brier,
    b1_brier,
    W2_brier,
    b2_brier
)

print("\nFinal train BCE:", Loss_final_brier[-1])
print("p_test_brier shape:", p_test_brier.shape)
print("p_test_brier min/max/mean:", p_test_brier.min(), p_test_brier.max(), p_test_brier.mean())

n_hidden: 2 | batch_size: 128 | eta: 1.0 | decay: 0.0 | validation Brier: 0.13962478212203547 | train BCE: 0.44523544438515755
n_hidden: 2 | batch_size: 128 | eta: 1.0 | decay: 0.001 | validation Brier: 0.13816364318941932 | train BCE: 0.44018369385681744
n_hidden: 2 | batch_size: 128 | eta: 1.0 | decay: 0.003 | validation Brier: 0.13781049432314982 | train BCE: 0.4390712825784284
n_hidden: 2 | batch_size: 128 | eta: 1.0 | decay: 0.01 | validation Brier: 0.13760005169901446 | train BCE: 0.4387691341749768
n_hidden: 2 | batch_size: 128 | eta: 0.3 | decay: 0.0 | validation Brier: 0.1386747990808042 | train BCE: 0.4455065092654426
n_hidden: 2 | batch_size: 128 | eta: 0.3 | decay: 0.001 | validation Brier: 0.1380772116234345 | train BCE: 0.44315734332117956
n_hidden: 2 | batch_size: 128 | eta: 0.3 | decay: 0.003 | validation Brier: 0.13779440666527365 | train BCE: 0.44234355258623964
n_hidden: 2 | batch_size: 128 | eta: 0.3 | decay: 0.01 | validation Brier: 0.1376901127197254 | train BCE: 

Here we have the model for the economic-loss model

In [ ]:
import numpy as np
from scipy.special import expit

# -----------------------------
# Helpers
# -----------------------------

def train_val_split(X, Y, val_fraction=0.2, seed=42):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    indices = rng.permutation(n)

    n_val = int(val_fraction * n)
    val_idx = indices[:n_val]
    train_idx = indices[n_val:]

    return X[train_idx], Y[train_idx], X[val_idx], Y[val_idx]


def fit_standardizer(X):
    mu = X.mean(axis=0, keepdims=True)
    sigma = X.std(axis=0, keepdims=True)
    sigma[sigma == 0] = 1.0
    return mu, sigma


def apply_standardizer(X, mu, sigma):
    return (X - mu) / sigma


def initialize_parameters(n_features, n_hidden, seed=42):
    rng = np.random.default_rng(seed)

    W1 = rng.normal(0.0, np.sqrt(2 / n_features), size=(n_features, n_hidden))
    b1 = np.zeros((1, n_hidden))

    W2 = rng.normal(0.0, np.sqrt(2 / n_hidden), size=(n_hidden, 1))
    b2 = np.zeros((1, 1))

    return W1, b1, W2, b2


def predict_neural_network(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1
    h = np.maximum(0, z1)

    z2 = h @ W2 + b2
    p = expit(z2)

    return p


def brier_score_weighted(Y, p):
    weights = np.where(Y == 1, 3.0, 1.0)
    return np.mean(weights * (Y - p) ** 2)


def weighted_binary_cross_entropy_from_logits(Y, z):
    # -mean(0.75 * Y * log(p) + 0.25 * (1 - Y) * log(1 - p))
    # Stable computation from logits, without taking log(0).
    return np.mean(
        0.75 * Y * np.logaddexp(0, -z)
        + 0.25 * (1 - Y) * np.logaddexp(0, z)
    )


def train_neural_network_minibatch_weighted_bce(
    X,
    Y,
    W1,
    b1,
    W2,
    b2,
    eta=0.01,
    decay=0.0,
    iterations=1000,
    batch_size=256,
    seed=42
):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    Loss = []

    for epoch in range(iterations):
        eta_epoch = eta / (1 + decay * epoch)

        indices = rng.permutation(n)

        for start in range(0, n, batch_size):
            batch_idx = indices[start:start + batch_size]

            X_batch = X[batch_idx]
            Y_batch = Y[batch_idx]
            m = X_batch.shape[0]

            # Forward pass
            z1 = X_batch @ W1 + b1
            h = np.maximum(0, z1)

            z2 = h @ W2 + b2
            p = expit(z2)

            # Weighted BCE backward pass, matching the 3/4 and 1/4 loss.
            weights_batch = np.where(Y_batch == 1, 0.75, 0.25)
            dz2 = weights_batch * (p - Y_batch) / m

            gradient_W2 = h.T @ dz2
            gradient_b2 = np.sum(dz2, axis=0, keepdims=True)

            relu_derivative = (z1 > 0).astype(float)
            dz1 = (dz2 @ W2.T) * relu_derivative

            gradient_W1 = X_batch.T @ dz1
            gradient_b1 = np.sum(dz1, axis=0, keepdims=True)

            # Update
            W1 -= eta_epoch * gradient_W1
            b1 -= eta_epoch * gradient_b1
            W2 -= eta_epoch * gradient_W2
            b2 -= eta_epoch * gradient_b2

        # Loss tracks weighted BCE; weighted Brier is used for validation.
        h_train = np.maximum(0, X @ W1 + b1)
        z_train = h_train @ W2 + b2
        Loss.append(weighted_binary_cross_entropy_from_logits(Y, z_train))

    return W1, b1, W2, b2, Loss


# -----------------------------
# 1. Train/validation split
# -----------------------------

# Keep labels as a column vector to match the predictions.
Y_trn = np.asarray(Y_trn, dtype=float).reshape(-1, 1)

X_fit_raw, Y_fit, X_val_raw, Y_val = train_val_split(
    X_trn,
    Y_trn,
    val_fraction=0.2,
    seed=42
)

# -----------------------------
# 2. Scaling fitted only on fit/train split
# -----------------------------

mu_val, sigma_val = fit_standardizer(X_fit_raw)

X_fit = apply_standardizer(X_fit_raw, mu_val, sigma_val)
X_val = apply_standardizer(X_val_raw, mu_val, sigma_val)

# -----------------------------
# 3. Hyperparameter search
# -----------------------------

eta_grid = [1.0, 0.3, 0.1, 0.03, 0.01]
decay_grid = [0.0, 0.001, 0.003, 0.01]
n_hidden_grid = [2, 4, 8, 16]
batch_size_grid = [128, 256, 512]

iterations = 1000
n_features = X_fit.shape[1]

results_weighted = []

for n_hidden in n_hidden_grid:
    for batch_size in batch_size_grid:
        for eta in eta_grid:
            for decay in decay_grid:
                W1, b1, W2, b2 = initialize_parameters(
                    n_features=n_features,
                    n_hidden=n_hidden,
                    seed=42
                )

                W1, b1, W2, b2, Loss = train_neural_network_minibatch_weighted_bce(
                    X_fit,
                    Y_fit,
                    W1,
                    b1,
                    W2,
                    b2,
                    eta=eta,
                    decay=decay,
                    iterations=iterations,
                    batch_size=batch_size,
                    seed=42
                )

                p_val = predict_neural_network(X_val, W1, b1, W2, b2)
                val_weighted = brier_score_weighted(Y_val, p_val)

                train_weighted_bce = Loss[-1]

                results_weighted.append({
                    "n_hidden": n_hidden,
                    "batch_size": batch_size,
                    "eta": eta,
                    "decay": decay,
                    "val_weighted": val_weighted,
                    "train_weighted_bce": train_weighted_bce
                })

                print(
                    "n_hidden:", n_hidden,
                    "| batch_size:", batch_size,
                    "| eta:", eta,
                    "| decay:", decay,
                    "| validation weighted Brier:", val_weighted,
                    "| train weighted BCE:", train_weighted_bce
                )

# -----------------------------
# 4. Select best hyperparameters
# -----------------------------

best_result_weighted = min(results_weighted, key=lambda x: x["val_weighted"])

best_n_hidden_weighted = best_result_weighted["n_hidden"]
best_batch_size_weighted = best_result_weighted["batch_size"]
best_eta_weighted = best_result_weighted["eta"]
best_decay_weighted = best_result_weighted["decay"]

print("\nBest weighted Brier result:")
print(best_result_weighted)

# -----------------------------
# 5. Final weighted BCE model, selected on validation weighted Brier
# -----------------------------

mu_final_weighted, sigma_final_weighted = fit_standardizer(X_trn)

X_trn_scaled_weighted = apply_standardizer(X_trn, mu_final_weighted, sigma_final_weighted)
X_test_scaled_weighted = apply_standardizer(X_test, mu_final_weighted, sigma_final_weighted)

W1_weighted, b1_weighted, W2_weighted, b2_weighted = initialize_parameters(
    n_features=X_trn_scaled_weighted.shape[1],
    n_hidden=best_n_hidden_weighted,
    seed=42
)

W1_weighted, b1_weighted, W2_weighted, b2_weighted, Loss_final_weighted = train_neural_network_minibatch_weighted_bce(
    X_trn_scaled_weighted,
    Y_trn,
    W1_weighted,
    b1_weighted,
    W2_weighted,
    b2_weighted,
    eta=best_eta_weighted,
    decay=best_decay_weighted,
    iterations=iterations,
    batch_size=best_batch_size_weighted,
    seed=42
)

p_test_weighted = predict_neural_network(
    X_test_scaled_weighted,
    W1_weighted,
    b1_weighted,
    W2_weighted,
    b2_weighted
)

print("\nFinal train weighted BCE:", Loss_final_weighted[-1])
print("p_test_weighted shape:", p_test_weighted.shape)
print("p_test_weighted min/max/mean:", p_test_weighted.min(), p_test_weighted.max(), p_test_weighted.mean())
